# AWS Job Offer Classifier Training Notebook

This notebook presents a simple educational ML workflow for classifying job offers by category and seniority level.

## 1. Project description

Goal:
- load a dataset locally or from S3,
- inspect class distributions,
- preprocess text,
- train two classifiers,
- evaluate results,
- save model artifacts for Lambda.

In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from src.preprocessing import clean_text

## 2. Load CSV from local path or S3 path

In [ ]:
local_path = Path('../data/raw/job_offers_hf_prepared.csv')
# Example S3 path for AWS usage:
# s3_path = 's3://your-bucket-name/raw/job_offers_hf_prepared.csv'

df = pd.read_csv(local_path)
df.head()

## 3. Basic dataset inspection

In [ ]:
print(df.shape)
print(df.isna().sum())
df.sample(5, random_state=42)

## 4. Category distribution

In [ ]:
df['category'].value_counts()

## 5. Level distribution

In [ ]:
df['level'].value_counts()

## 6. Preprocessing

We apply a small cleaning function that keeps technical terms such as `c++`, `c#`, and `node.js`.

In [ ]:
df['clean_text'] = df['text'].fillna('').map(clean_text)
df[['text', 'clean_text']].head(3)

## 7. Train/test split

In [ ]:
X_train, X_test, y_category_train, y_category_test, y_level_train, y_level_test = train_test_split(
    df['clean_text'],
    df['category'],
    df['level'],
    test_size=0.2,
    random_state=42,
    stratify=df[['category', 'level']]
)
len(X_train), len(X_test)

## 8. TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
X_train_vec.shape

## 9. Train models

In [ ]:
category_model = LogisticRegression(max_iter=1000, random_state=42)
level_model = LogisticRegression(max_iter=1000, random_state=42)

category_model.fit(X_train_vec, y_category_train)
level_model.fit(X_train_vec, y_level_train)

## 10. Evaluation

In [ ]:
category_predictions = category_model.predict(X_test_vec)
level_predictions = level_model.predict(X_test_vec)

print('Category report')
print(classification_report(y_category_test, category_predictions, zero_division=0))
print('Level report')
print(classification_report(y_level_test, level_predictions, zero_division=0))

## 11. Save model artifacts

In [ ]:
model_dir = Path('../model')
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, model_dir / 'vectorizer.joblib')
joblib.dump(category_model, model_dir / 'category_model.joblib')
joblib.dump(level_model, model_dir / 'level_model.joblib')

labels = {
    'categories': sorted(df['category'].unique().tolist()),
    'levels': sorted(df['level'].unique().tolist()),
}

with (model_dir / 'labels.json').open('w', encoding='utf-8') as file:
    json.dump(labels, file, indent=2)

print('Artifacts saved to', model_dir)

## 12. Explanation of results

The synthetic dataset contains clearly separated keywords, so the model should achieve strong results. In a real-world project, performance would likely be lower and require more advanced preprocessing and a more diverse dataset.